# Logistic Regression

This notebook will perform a logistic regression on different features of the collected data, in order to see if there are any patterns on which restaurants were easy or difficult to find a menu from.

## The Dataset

The dataset consists of responses from two LLMs. Process the [Prompt](../../prompts/improved-20260910.md) to understand what they were instructed in.
The input datset was [copenhagen-bounds-full-2026-10-01.json](../../data/raw_data/copenhagen-bounds-full-2026-10-01.json)
The output dataset was [this for Terra](../../data/api_runs/responses/20261001-133048-gpt-5.6-terra.csv) and [this for Luna](../../data/api_runs/responses/20261002-094032-gpt-5.6-luna.csv)
The full (raw) dataset, with more parameters than the filtered ones, is [this dataset (GoogleMaps API raw](../../data/raw_data/copenhagen-bounds-full-raw.json)

## Model Agreement

Each model now tried to find a menu for all the restaurant. It turns out that they didn't agree that well; only on 52% of the restaurants, they found exactly the same menu link. So I would like to inspect further, what might have triggered them to having difficulties in 1) locating different menu URLS and 2) not being able to find a menu.

## The Logistic Regression

I would like to further inspect if there are other features that might have influenced why they couldn't find a menu URL. So for the regression, the dependent variable is `hasFoundMenu` and the independent variables are other factors that might be present.

Out of the factors that might be present, I would like to inspect if the restaruant:

- has a website? 
- has any other tags, or does it only have a single tag?
- reviews?
- also tagged with takeaway or delievry / offers takeaway?
- also tagged with other food types, such as café, bar, pub, bakery. Refer to [Google Maps API's Food And Drink Section](https://developers.google.com/maps/documentation/places/web-service/place-types#food-and-drink)
- price?

In [13]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from data_cleaner import *
from config import *
pd.set_option("display.max_columns", None)

In [14]:
# shared plot config, same as fulldataset_analysis.ipynb
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

thesis_template = go.layout.Template(pio.templates["plotly"])
thesis_template.layout.update(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="white", font=dict(size=15))
axis_style = dict(showline=True, linecolor="black", linewidth=1, ticks="outside", tickcolor="black", ticklen=5,
                  showgrid=False, zeroline=False, mirror=False)
thesis_template.layout.xaxis.update(axis_style)
thesis_template.layout.yaxis.update(axis_style)
pio.templates["thesis"] = thesis_template
pio.templates.default = "thesis"

MODEL_NAMES = ['claude-sonnet-5', 'gemini-3.7-flash', 'gpt-5.6-luna', 'gpt-5.6-terra']
MODEL_COLORS = dict(zip(MODEL_NAMES, px.colors.qualitative.Plotly))
MARGIN = {"r": 20, "t": 80, "l": 20, "b": 60}

def plot_title(title, subtitle=None):
    return f"<b>{title}</b>" + (f"<br><sup>{subtitle}</sup>" if subtitle else "")

def save_plot(fig, filename):
    fig.write_image(PLOTS_DIR / filename)

## Load the data

- **Outcomes**: the Terra and Luna responses (one row per restaurant per model)
- **Inputs**: the raw Google Maps data, filtered the same way as the input dataset (operational + restaurant primary type), which gives the same 3,125 restaurants

In [15]:
df = get_df(API_DATA_DIR / 'responses', '2026100*')   # terra + luna, the 'deprecated-' luna run is skipped
places = filter_dataframe(pd.read_json(RAW_DATA_DIR / 'copenhagen-bounds-full-raw.json')).drop_duplicates('id').set_index('id')

print(f"{len(places)} restaurants, {df['model'].nunique()} models, {len(df)} responses")

3125 restaurants, 2 models, 6250 responses


## Independent variables

One row per restaurant, built from the questions in the intro:

| Question | Variable(s) |
|---|---|
| has a website? | `has website` |
| any other tags, or only a single tag? | `only generic tag`: the `types` list holds nothing but `restaurant`, `food`, `point_of_interest`, `establishment` |
| reviews? | `reviews (log10)` = log10(review count + 1), `no reviews` |
| takeaway / delivery? | `offers takeout`, `offers delivery` (Google attributes `takeout` / `delivery`; missing counts as no) |
| reservations? | `offers reservations` (Google attribute `reservable`; missing counts as no) |
| other food types (café, bar, pub, bakery, ...)? | `also tagged: ...`, the [Food and Drink](https://developers.google.com/maps/documentation/places/web-service/place-types#food-and-drink) types that are not restaurant types, grouped by kind of place |
| pizza, burger, kebab places? | `also tagged: pizza`, `also tagged: burger/american`, `also tagged: kebab/halal` (restaurant types) |
| price? | `price: inexpensive`, `price: expensive`, `price: unknown` (moderate is the reference). `priceLevel` is missing for 1,859 restaurants, so for those it is inferred from `priceRange` (see below) |

`reviews (log10)` is 0 for restaurants without reviews; the `no reviews` dummy lets those restaurants differ from what the log scale alone would predict.

### Inferring the price level from the price range

Google gives `priceRange` (DKK per person, e.g. 100–200) for some restaurants that have no `priceLevel`. Among the restaurants that have both, each range is mapped to the price level most of them have (expensive and very expensive are merged). A range that never occurs together with a price level gets the level whose median range midpoint is closest. The mapping is not clean (e.g. 1–100 DKK is split almost evenly between inexpensive and moderate), so the printed agreement says how often it matches the real `priceLevel`.

In [16]:
PRICE_LEVELS = {'PRICE_LEVEL_INEXPENSIVE': 'inexpensive', 
                'PRICE_LEVEL_MODERATE': 'moderate',
                'PRICE_LEVEL_EXPENSIVE': 'expensive', 
                'PRICE_LEVEL_VERY_EXPENSIVE': 'expensive'}

def range_key(r):      # e.g. '100-200', or '1000-' for the open-ended range
    return f"{r['startPrice']['units']}-{r.get('endPrice', {}).get('units', '')}" if isinstance(r, dict) else None

def range_mid(r):
    if not isinstance(r, dict):
        return np.nan
    start = float(r['startPrice']['units'])
    return (start + float(r.get('endPrice', {}).get('units', start))) / 2

price_known = places['priceLevel'].map(PRICE_LEVELS)
price_range = places['priceRange'].apply(range_key)
price_mid = places['priceRange'].apply(range_mid)

labelled = price_known.notna() & price_range.notna()
range_to_level = pd.crosstab(price_range[labelled], price_known[labelled]).idxmax(axis=1)
level_mid = price_mid[labelled].groupby(price_known[labelled]).median()

price_inferred = price_range.map(range_to_level)
unseen = price_range.notna() & price_inferred.isna()
price_inferred[unseen] = price_mid[unseen].apply(lambda mid: (level_mid - mid).abs().idxmin())
price = price_known.fillna(price_inferred)

print(f"inferred matches the real priceLevel for {(price_inferred[labelled] == price_known[labelled]).mean():.1%} of the {labelled.sum()} restaurants that have both")
print(f"price known: {price_known.notna().sum()}, inferred: {(price_known.isna() & price.notna()).sum()}, still unknown: {price.isna().sum()}")
pd.crosstab(price_range[labelled], price_known[labelled]).assign(inferred=range_to_level)[['inexpensive', 'moderate', 'expensive', 'inferred']]

inferred matches the real priceLevel for 79.6% of the 1099 restaurants that have both
price known: 1266, inferred: 1185, still unknown: 674


priceLevel,inexpensive,moderate,expensive,inferred
priceRange,,,,
1-100,125,110,0,inexpensive
1-200,1,5,0,moderate
1-300,1,9,0,moderate
1-400,1,5,0,moderate
1-500,0,1,0,moderate
1-600,0,3,0,moderate
100-200,72,333,0,moderate
100-400,1,1,0,inexpensive
100-500,0,8,0,moderate


In [17]:
GENERIC_TYPES = {'restaurant', 'food', 'point_of_interest', 'establishment'}

# non-restaurant Food and Drink types, grouped by kind of place (same groups as fulldataset_analysis.ipynb,
# except kebab/shawarma, which moved from quick food to the kebab/halal group), then three restaurant-type groups
TAG_GROUPS = {
    '+ tagged: bar/pub':         ['bar', 'wine_bar', 'cocktail_bar', 'bar_and_grill', 'pub', 'irish_pub', 'gastropub', 'lounge_bar',
                                     'sports_bar', 'hookah_bar', 'beer_garden', 'brewery', 'brewpub', 'winery'],
    '+ tagged: cafe':            ['cafe', 'coffee_shop', 'cafeteria', 'coffee_roastery', 'coffee_stand', 'tea_house', 'cat_cafe', 'dog_cafe'],
    '+ tagged: bakery/sweets':   ['bakery', 'cake_shop', 'pastry_shop', 'confectionery', 'dessert_shop', 'ice_cream_shop', 'donut_shop',
                                     'chocolate_shop', 'chocolate_factory', 'candy_store', 'juice_shop', 'acai_shop'],
    '+ tagged: quick food':      ['sandwich_shop', 'salad_shop', 'hot_dog_restaurant', 'hot_dog_stand',
                                     'snack_bar', 'noodle_shop', 'deli', 'food_court'],
    '+ tagged: pizza':           ['pizza_restaurant', 'pizza_delivery'],
    '+ tagged: burger/american': ['hamburger_restaurant', 'american_restaurant', 'diner'],
    '+ tagged: kebab/halal':     ['kebab_shop', 'shawarma_restaurant', 'falafel_restaurant', 'halal_restaurant',
                                     'turkish_restaurant', 'middle_eastern_restaurant', 'lebanese_restaurant'],
}

# helpers
tags = places['types'].apply(lambda ts: set(ts) - GENERIC_TYPES)
has_tag = lambda names: tags.apply(lambda t: bool(t & set(names)))
is_yes = lambda col: places[col].eq(1)    # missing counts as "not yes"

# build features
features = pd.DataFrame({
    'has website':              places['websiteUri'].notna(),
    'only generic tag':         tags.str.len().eq(0),
    'reviews (log10)':          np.log10(places['userRatingCount'].fillna(0) + 1),
    'no reviews':               places['userRatingCount'].isna(),
    'offers takeout':           is_yes('takeout'),
    'offers delivery':          is_yes('delivery'),
    'offers reservations':      is_yes('reservable'),
    # for each TAG_GROUPS, check if the remaining tags belong to any of them
    **{group: has_tag(names) for group, names in TAG_GROUPS.items()},
    'price: inexpensive':       price.eq('inexpensive'),
    'price: expensive':         price.eq('expensive'),
    'price: unknown':           price.isna(),
}).astype(float)

features.describe().T[['mean', 'min', 'max']].round(2)   # mean = share of restaurants for the yes/no inputs

,mean,min,max
has website,0.82,0.0,1.00
only generic tag,0.33,0.0,1.00
reviews (log10),1.96,0.0,3.93
no reviews,0.09,0.0,1.00
offers takeout,0.69,0.0,1.00
offers delivery,0.32,0.0,1.00
offers reservations,0.53,0.0,1.00
+ tagged: bar/pub,0.04,0.0,1.00
+ tagged: cafe,0.04,0.0,1.00
+ tagged: bakery/sweets,0.01,0.0,1.00


In [18]:
features.head()

,has website,only generic tag,reviews (log10),no reviews,offers takeout,offers delivery,offers reservations,+ tagged: bar/pub,+ tagged: cafe,+ tagged: bakery/sweets,+ tagged: quick food,+ tagged: pizza,+ tagged: burger/american,+ tagged: kebab/halal,price: inexpensive,price: expensive,price: unknown
id,,,,,,,,,,,,,,,,,
ChIJ6eXKTWdDUkYR7k5XjV6Ju-0,1.0,0.0,3.138934,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ChIJ21qBwtRDUkYRGBAxnmFfg10,1.0,1.0,2.146128,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ChIJV5-mcl5DUkYRbdj-CHc26vo,1.0,0.0,2.012837,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
ChIJIbl0IgBFUkYRU0k537QppSU,0.0,1.0,0.698970,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
ChIJt4pwV39FUkYRE1tcx-SMAqY,0.0,1.0,0.301030,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


### Do the inputs overlap?

Strongly correlated inputs make the individual odds ratios unstable (wide confidence intervals), so check this before reading the regression. The variance inflation factor (VIF) says how much each coefficient's variance is inflated by the other inputs; above ~5 is worth worrying about.

In [19]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

corr = features.corr()
pairs = corr.where(np.tril(np.ones(corr.shape, dtype=bool), k=-1)).stack()
print("pairs with |r| >= 0.4:")
print(pairs[pairs.abs() >= 0.4].sort_values(key=abs, ascending=False).round(2).to_string())

X = sm.add_constant(features)
vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])], index=features.columns, name='VIF')
vif.sort_values(ascending=False).round(2)

pairs with |r| >= 0.4:
price: unknown       reviews (log10)   -0.73
no reviews           reviews (log10)   -0.67
price: unknown       no reviews         0.60
reviews (log10)      has website        0.57
price: unknown       has website       -0.49
offers reservations  reviews (log10)    0.47
price: unknown       offers takeout    -0.46
offers delivery      offers takeout     0.44
no reviews           has website       -0.43


reviews (log10)              3.44
price: unknown               2.76
no reviews                   1.95
offers takeout               1.68
has website                  1.61
offers reservations          1.58
only generic tag             1.44
offers delivery              1.35
price: inexpensive           1.29
+ tagged: pizza              1.26
+ tagged: burger/american    1.14
price: expensive             1.14
+ tagged: quick food         1.13
+ tagged: bakery/sweets      1.13
+ tagged: bar/pub            1.11
+ tagged: kebab/halal        1.08
+ tagged: cafe               1.07
Name: VIF, dtype: float64

## Dependent variable: `both found`

The two models are not analysed separately. The outcome is `both found` = 1 when **both** models found a menu for the restaurant, and 0 otherwise, i.e. when neither found one or only one of them did. All 3,125 restaurants are included.

In [20]:
found = df.pivot(index='id', columns='model', values='hasFoundMenu').astype(bool).loc[features.index]
y = found.all(axis=1).astype(float).rename('both found')

print(pd.crosstab(found['gpt-5.6-luna'], found['gpt-5.6-terra'], margins=True))
print(f"both found: {int(y.sum())} of {len(y)} restaurants ({y.mean():.1%})")

gpt-5.6-terra  False  True   All
gpt-5.6-luna                    
False            617   299   916
True             119  2090  2209
All              736  2389  3125
both found: 2090 of 3125 restaurants (66.9%)


## The logistic regression

`statsmodels` Logit, all inputs at once, so each odds ratio is the effect of that input *holding the others constant*.
Odds ratio **> 1 = both models more likely to find a menu**, < 1 = less likely.

In [21]:
def fit_logit(y, X):
    fit = sm.Logit(y, sm.add_constant(X)).fit(disp=0)
    ci = fit.conf_int()
    table = pd.DataFrame({
        'odds ratio': np.exp(fit.params),
        'ci low':     np.exp(ci[0]),
        'ci high':    np.exp(ci[1]),
        'p':          fit.pvalues,
    }).drop(index='const')
    return fit, table

fit, result = fit_logit(y, features)
print(f"n = {int(fit.nobs)}, both found = {y.mean():.1%}, pseudo R² = {fit.prsquared:.3f}, LR test p = {fit.llr_pvalue:.1e}")
result.round(3)

n = 3125, both found = 66.9%, pseudo R² = 0.263, LR test p = 4.3e-211


,odds ratio,ci low,ci high,p
has website,4.283,3.266,5.617,0.000
only generic tag,0.795,0.636,0.994,0.044
reviews (log10),1.811,1.518,2.161,0.000
no reviews,0.568,0.343,0.942,0.028
offers takeout,1.125,0.879,1.439,0.350
offers delivery,1.501,1.196,1.884,0.000
offers reservations,1.376,1.101,1.720,0.005
+ tagged: bar/pub,0.740,0.474,1.155,0.184
+ tagged: cafe,0.722,0.447,1.164,0.181
+ tagged: bakery/sweets,0.750,0.363,1.552,0.439


In [22]:
# full statsmodels summary (coefficients are log-odds, not odds ratios)
fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:             both found   No. Observations:                 3125
Model:                          Logit   Df Residuals:                     3107
Method:                           MLE   Df Model:                           17
Date:                Thu, 08 Oct 2026   Pseudo R-squ.:                  0.2629
Time:                        17:05:49   Log-Likelihood:                -1462.7
converged:                       True   LL-Null:                       -1984.5
Covariance Type:            nonrobust   LLR p-value:                4.315e-211
=============================================================================================
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                        -1.6623      0.238     -6.982      0.000      -2.129      -1.196
has website                   1.4547      0.138     10.518      0.000       1.184       1.726
only generic tag             -0.2288      0.114     -2.011      0.044      -0.452      -0.006
reviews (log10)               0.5939      0.090      6.589      0.000       0.417       0.771
no reviews                   -0.5655      0.258     -2.193      0.028      -1.071      -0.060
offers takeout                0.1174      0.126      0.934      0.350      -0.129       0.364
offers delivery               0.4062      0.116      3.508      0.000       0.179       0.633
offers reservations           0.3192      0.114      2.807      0.005       0.096       0.542
+ tagged: bar/pub            -0.3017      0.227     -1.327      0.184      -0.747       0.144
+ tagged: cafe               -0.3264      0.244     -1.338      0.181      -0.804       0.152
+ tagged: bakery/sweets      -0.2870      0.371     -0.774      0.439      -1.013       0.439
+ tagged: quick food         -0.1475      0.312     -0.472      0.637      -0.760       0.465
+ tagged: pizza              -0.1123      0.142     -0.789      0.430      -0.391       0.167
+ tagged: burger/american     0.4116      0.196      2.098      0.036       0.027       0.796
+ tagged: kebab/halal        -0.1968      0.237     -0.831      0.406      -0.661       0.268
price: inexpensive           -0.1688      0.130     -1.302      0.193      -0.423       0.085
price: expensive              0.1648      0.264      0.624      0.533      -0.353       0.683
price: unknown               -0.5185      0.165     -3.147      0.002      -0.841      -0.196
=============================================================================================
"""

In [26]:
# forest plot: odds ratio with 95% CI per input, grouped by kind of feature with short labels
REFERENCE = 'price: moderate'    # not a feature: the reference category the other price levels are compared with, OR = 1
PLOT_GROUPS = {
    'Online presence':      {'has website': 'Has website'},
    'Reviews':              {'reviews (log10)': '10× more reviews', 'no reviews': 'No reviews'},
    'Services':             {'offers takeout': 'Takeout', 'offers delivery': 'Delivery', 'offers reservations': 'Reservations'},
    'Google tags':          {'only generic tag': 'Only generic tags',
                             # 'tagged: bar/pub' -> 'Bar / pub'
                             **{group: group.split(': ', 1)[1].replace('/', ' / ').capitalize() for group in TAG_GROUPS}},
    'Price':                {'price: inexpensive': 'Inexpensive', REFERENCE: 'Moderate (reference)',
                             'price: expensive': 'Expensive', 'price: unknown': 'Unknown'},
}
assert sorted(f for g in PLOT_GROUPS.values() for f in g if f != REFERENCE) == sorted(result.index)

# one row per header and per feature, top to bottom
rows = []
for group, labels in PLOT_GROUPS.items():
    rows.append((f"<b>{group}</b>", None))
    rows += [(label, feature) for feature, label in labels.items()]
y_of = {feature: -i for i, (_, feature) in enumerate(rows) if feature}

res = result.loc[[f for f in y_of if f != REFERENCE]]
significant = (res["ci low"] > 1) | (res["ci high"] < 1)
INK, MUTED = "#222222", "#9e9e9e"
colors = np.where(significant, INK, MUTED)
labels = {feature: label for label, feature in rows if feature}

fig = go.Figure(go.Scatter(
    x=res["odds ratio"],
    y=[y_of[f] for f in res.index],
    mode="markers",
    marker=dict(color=colors, size=10),
    error_x=dict(type="data", symmetric=False,
                 array=res["ci high"] - res["odds ratio"],
                 arrayminus=res["odds ratio"] - res["ci low"],
                 thickness=2, width=0, color=MUTED),
    customdata=np.stack([res["ci low"], res["ci high"], res["p"]], axis=-1),
    text=[labels[f] for f in res.index],
    hovertemplate="<b>%{text}</b><br>OR %{x:.2f} (95% CI %{customdata[0]:.2f}, %{customdata[1]:.2f})<br>p = %{customdata[2]:.3f}<extra></extra>",
    showlegend=False,
))

# odds ratio and CI as a text column to the right of the plot
for (feature, row), color in zip(res.iterrows(), colors):
    fig.add_annotation(x=1.02, xref="paper", y=y_of[feature], yref="y", xanchor="left", showarrow=False,
                       text=f"{row['odds ratio']:.2f}  ({row['ci low']:.2f}, {row['ci high']:.2f})", font=dict(color=color, size=14))
fig.add_annotation(x=1.02, xref="paper", y=1, yref="paper", yanchor="bottom", xanchor="left", showarrow=False,
                   text="<b>OR  (95% CI)</b>", font=dict(size=14))

# reference category: fixed at OR = 1, no CI
fig.add_trace(go.Scatter(x=[1], y=[y_of[REFERENCE]], mode="markers", showlegend=False,
                         marker=dict(symbol="diamond", size=11, color=INK),
                         hovertemplate="<b>Moderate</b><br>reference category (OR = 1)<extra></extra>"))
fig.add_annotation(x=1.02, xref="paper", y=y_of[REFERENCE], yref="y", xanchor="left", showarrow=False,
                   text="1.00  (reference)", font=dict(color=INK, size=14))

fig.add_vline(x=1, line=dict(color=INK, width=1, dash="dot"))
fig.update_layout(
    title=plot_title("What makes a menu easy to find?",
                     f"outcome: both models found a menu · odds ratio with 95% CI · > 1 = more likely · grey = CI includes 1 · n = {len(features):,} restaurants"),
    xaxis=dict(type="log", title="Odds ratio (log scale)",
               tickvals=[0.25, 0.5, 1, 2, 4, 8], ticktext=["0.25", "0.5", "1", "2", "4", "8"]),
    yaxis=dict(title=None, tickvals=[-i for i in range(len(rows))], ticktext=[label for label, _ in rows],
               range=[-len(rows) + 0.5, 0.5], showgrid=True, gridcolor="#eeeeee", ticks=""),
    height=810, width=1050,
    margin={**MARGIN, "l": 190, "r": 200, "t": 100},
)

save_plot(fig, 'logistic-regression-hasfoundmenu-full.svg')
fig.show()

## Follow-up: different menu links

The intro also asks what makes the models find *different* menu links. Among the restaurants where **both** found a menu, `same link` = the two `menuLink`s are equal after normalising them with `trim_urls` (drops `http(s)://`, `www.` and a trailing `/`, so `https://www.x.dk/menu/` and `x.dk/menu` count as the same).
Same inputs, same model. Odds ratio > 1 = more likely to agree on the link.

In [24]:
links = trim_urls(df.copy()).pivot(index='id', columns='model', values='normalized_urls').loc[features.index]
both = y.astype(bool)
same_link = (links.loc[both, 'gpt-5.6-luna'] == links.loc[both, 'gpt-5.6-terra']).astype(float)
print(f"both found: {both.sum()} restaurants, same link on {same_link.mean():.1%} of them")

# 'no reviews' / 'price: unknown' etc. can become (near) constant in the subset, drop inputs without variation
X_both = features.loc[both]
X_both = X_both.loc[:, X_both.std() > 0]
fit_same, same_link_result = fit_logit(same_link, X_both)
print(f"pseudo R² = {fit_same.prsquared:.3f}, LR test p = {fit_same.llr_pvalue:.1e}")
same_link_result.sort_values('odds ratio').round(3)

both found: 2090 restaurants, same link on 48.3% of them
pseudo R² = 0.022, LR test p = 1.8e-07


,odds ratio,ci low,ci high,p
offers reservations,0.743,0.600,0.920,0.006
offers delivery,0.805,0.656,0.989,0.039
reviews (log10),0.865,0.729,1.025,0.095
+ tagged: bar/pub,0.878,0.574,1.342,0.548
+ tagged: burger/american,0.908,0.676,1.221,0.524
price: unknown,0.925,0.631,1.356,0.690
+ tagged: quick food,0.951,0.537,1.684,0.863
offers takeout,0.962,0.742,1.246,0.767
only generic tag,0.962,0.766,1.208,0.739
+ tagged: bakery/sweets,0.991,0.466,2.108,0.982


## Findings (run of 2026-10-07)

**Finding a menu (`hasFoundMenu`)**. The model explains a fair amount (pseudo R² 0.26–0.36, in-sample ROC AUC 0.80–0.86). No input overlaps too much (all VIF < 4). Odds ratios are for `both found`:

- **Website** is by far the strongest predictor. With a website, the odds that both models find a menu are about 4.7× higher.
- **Reviews**: each 10× more reviews gives about 2.3× higher odds, and no reviews at all gives about 0.5×. The star rating itself does not matter.
- **Takeaway / delivery**: only the Google *attribute* `offers delivery` helps (≈1.5×), probably because those places are on Wolt or Just Eat. The type tag and `offers takeout` are not significant.
- **Only generic tag**: lower odds. This is clear for Terra (0.62×) but borderline for both found (0.81×, p = 0.06).
- **Other food types** (bar/pub, café, bakery): the odds are below 1 for all of them, but they are only significant for Terra. These groups are small, at 1–4% of the restaurants.
- **Price**: `inexpensive` gives lower odds (≈0.68×). Expensive and unknown prices make no difference.

**Finding the same link (given both found a menu)**. After `trim_urls`, the models agree on 48.3% of the 2,090 restaurants. The inputs barely explain this (pseudo R² 0.01, AUC 0.56). The disagreement on links is mostly *not* explained by these listing features. It is more likely about which page/format each model chose (see the source/format analysis).